# Meta-Learning for Few-Shot Learning

## What We'll Learn

In this notebook, we'll explore **meta-learning** — the fascinating paradigm of "learning to learn". Unlike traditional machine learning where we train models on large datasets, meta-learning trains models that can quickly adapt to new tasks with just a few examples.

**Key concepts we'll build:**

- The **few-shot learning problem**: learning from 1-5 examples per class
- **N-way K-shot classification**: the standard meta-learning setup
- **Support and query sets**: how meta-learning splits data
- **MAML** (Model-Agnostic Meta-Learning): learning initial parameters that adapt quickly
- **Inner and outer loops**: the two-level optimization of meta-learning
- **Second-order gradients**: computing gradients through gradients
- Alternative approaches: **Prototypical Networks** and **Matching Networks**

**Why this matters:**

Meta-learning addresses a fundamental limitation of deep learning: the need for large amounts of labeled data. By learning how to learn, meta-learning models can:
- Adapt to new tasks with minimal examples (like humans do)
- Generalize across diverse task distributions
- Enable practical applications where labeled data is expensive or scarce

We'll implement MAML from scratch for sine wave regression, then explore metric-based approaches that learn better similarity functions.

## 1. Setup

### Import libraries and configure environment

We'll use PyTorch for implementation and the shared library for utilities.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict
from copy import deepcopy

from aiml_notebooks import get_device, set_seed

### Set random seed for reproducibility

In [ ]:
set_seed(42)

### Configure device

We'll use CPU for this notebook since meta-learning involves many small batches and second-order gradients.

In [ ]:
device = get_device(prefer_cpu=True)

## 2. Understanding the Few-Shot Learning Problem

### What is Few-Shot Learning?

Traditional supervised learning requires hundreds or thousands of examples per class. **Few-shot learning** tackles the challenge of learning from just a few examples — often 1-5 per class.

The standard formulation is **N-way K-shot classification**:
- **N-way**: classify among N classes
- **K-shot**: only K labeled examples per class

For example, "5-way 1-shot" means: given 1 example each of 5 different classes, classify new examples.

**The key insight:** Instead of training on a single large dataset, we train on many small tasks. The model learns to adapt quickly from this **distribution of tasks**.

### Support Set vs Query Set

Meta-learning uses a different data split than traditional ML:

- **Support set**: The K examples per class used for adaptation (like training data)
- **Query set**: Examples used to evaluate adaptation (like test data)

Each **task** or **episode** has its own support and query sets. During meta-training, we sample many such tasks.

Let's visualize this concept with a simple example.

In [ ]:
# Simulate a 3-way 2-shot classification task
n_way = 3
k_shot = 2
n_query = 3

# Generate synthetic data points for visualization
np.random.seed(42)
classes = ['Class A', 'Class B', 'Class C']
colors = ['red', 'blue', 'green']

plt.figure(figsize=(12, 5))

# Support set
plt.subplot(1, 2, 1)
for i, (cls, color) in enumerate(zip(classes, colors)):
    # Generate support examples
    support_x = np.random.randn(k_shot, 2) + [i*3, 0]
    plt.scatter(support_x[:, 0], support_x[:, 1], 
               c=color, marker='o', s=200, edgecolors='black', linewidth=2,
               label=f'{cls} (support)')
    
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title(f'Support Set ({n_way}-way {k_shot}-shot)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Query set
plt.subplot(1, 2, 2)
for i, (cls, color) in enumerate(zip(classes, colors)):
    # Generate query examples
    query_x = np.random.randn(n_query, 2) + [i*3, 0]
    plt.scatter(query_x[:, 0], query_x[:, 1], 
               c=color, marker='x', s=200, linewidth=3,
               label=f'{cls} (query)')
    
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title(f'Query Set (evaluation)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Support set: {n_way} classes × {k_shot} examples = {n_way * k_shot} total examples")
print(f"Query set: {n_way} classes × {n_query} examples = {n_way * n_query} total examples")
print("\nThe model adapts using the support set (●) and is evaluated on the query set (×)")

### The Meta-Learning Setup

Meta-learning operates at two levels:

1. **Inner loop** (task-level): Fast adaptation to a specific task using the support set
2. **Outer loop** (meta-level): Learning across tasks to improve adaptation ability

The goal is to find model parameters $\theta$ that, when adapted to a new task with a few gradient steps, achieve good performance.

**Key difference from transfer learning:**
- Transfer learning: pre-train on source task → fine-tune on target task (one-time)
- Meta-learning: train to adapt quickly across a distribution of tasks (many times)

## 3. MAML: Model-Agnostic Meta-Learning

### The MAML Algorithm

**MAML** (Finn et al., 2017) learns initial parameters $\theta$ that are easy to fine-tune. The key idea:

Find $\theta$ such that one or a few gradient steps lead to good performance on new tasks.

**Algorithm:**

For each meta-training iteration:
1. Sample a batch of tasks $\mathcal{T}_i$
2. For each task $\mathcal{T}_i$:
   - **Inner loop**: Adapt $\theta$ using support set: $\theta'_i = \theta - \alpha \nabla_{\theta} \mathcal{L}_{\mathcal{T}_i}^{\text{support}}(\theta)$
   - Compute loss on query set using adapted parameters: $\mathcal{L}_{\mathcal{T}_i}^{\text{query}}(\theta'_i)$
3. **Outer loop**: Update meta-parameters: $\theta \leftarrow \theta - \beta \nabla_{\theta} \sum_i \mathcal{L}_{\mathcal{T}_i}^{\text{query}}(\theta'_i)$

The crucial detail: we compute **gradients through gradients** — the outer loop optimizes the post-adaptation performance.

### Sine Wave Regression Task

We'll implement MAML on a simple regression problem: fitting sine waves with different amplitudes and phases.

Each **task** is a different sine wave: $y = a \sin(x + b)$, where $a$ (amplitude) and $b$ (phase) vary.

This is a perfect toy problem because:
- Each task is simple but different
- We can visualize the adaptation process
- It demonstrates the core MAML principles

### Generate Sine Wave Tasks

Let's create a task generator that samples random sine waves.

In [ ]:
class SineWaveTask:
    """A single sine wave regression task."""
    
    def __init__(self, amplitude: float = None, phase: float = None):
        # Random amplitude and phase if not specified
        self.amplitude = amplitude if amplitude is not None else np.random.uniform(0.1, 5.0)
        self.phase = phase if phase is not None else np.random.uniform(0, np.pi)
        
    def sample(self, n_samples: int, x_range: Tuple[float, float] = (-5, 5)) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample n points from this sine wave."""
        x = np.random.uniform(x_range[0], x_range[1], n_samples)
        y = self.amplitude * np.sin(x + self.phase)
        return torch.tensor(x, dtype=torch.float32).unsqueeze(1), torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    
    def true_function(self, x: np.ndarray) -> np.ndarray:
        """Compute the true function for visualization."""
        return self.amplitude * np.sin(x + self.phase)

### Visualize different sine wave tasks

In [ ]:
# Create 4 random tasks
np.random.seed(42)
tasks = [SineWaveTask() for _ in range(4)]

plt.figure(figsize=(14, 8))
x_plot = np.linspace(-5, 5, 200)

for idx, task in enumerate(tasks, 1):
    plt.subplot(2, 2, idx)
    
    # Sample support and query points
    x_support, y_support = task.sample(10)
    x_query, y_query = task.sample(10)
    
    # Plot true function
    y_true = task.true_function(x_plot)
    plt.plot(x_plot, y_true, 'k-', linewidth=2, label='True function')
    
    # Plot support and query points
    plt.scatter(x_support.numpy(), y_support.numpy(), 
               c='blue', marker='o', s=100, edgecolors='black', linewidth=1.5,
               label='Support set', zorder=3)
    plt.scatter(x_query.numpy(), y_query.numpy(), 
               c='red', marker='x', s=100, linewidth=2,
               label='Query set', zorder=3)
    
    plt.xlabel('x', fontsize=11)
    plt.ylabel('y', fontsize=11)
    plt.title(f'Task {idx}: a={task.amplitude:.2f}, phase={task.phase:.2f}', fontsize=12, fontweight='bold')
    plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.ylim(-6, 6)

plt.tight_layout()
plt.show()

print("Each task is a different sine wave with random amplitude and phase.")
print("The model must learn to quickly adapt to any new sine wave using just the support set (blue dots).")

### Define the Model

We'll use a simple 3-layer MLP. The model architecture doesn't matter much — MAML is model-agnostic!

In [ ]:
class SineModel(nn.Module):
    """Simple MLP for sine wave regression."""
    
    def __init__(self, hidden_size: int = 40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
        
    def forward(self, x):
        return self.net(x)

### Create model and inspect architecture

In [ ]:
model = SineModel(hidden_size=40).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Implementing MAML from Scratch

### Inner Loop: Task Adaptation

The **inner loop** performs fast adaptation to a single task. Given support set $(x_s, y_s)$:

$$\theta' = \theta - \alpha \nabla_{\theta} \mathcal{L}_{\text{support}}(\theta)$$

We'll implement this as one or more gradient descent steps.

**Critical PyTorch detail:** We must use `create_graph=True` when computing gradients in the inner loop, so PyTorch builds a computation graph for the outer loop gradient computation.

In [ ]:
def inner_loop_adapt(
    model: nn.Module,
    x_support: torch.Tensor,
    y_support: torch.Tensor,
    inner_lr: float,
    inner_steps: int,
    create_graph: bool = True
) -> List[torch.Tensor]:
    """
    Perform inner loop adaptation on a single task.
    
    Args:
        model: The model to adapt
        x_support: Support set inputs (K, 1)
        y_support: Support set outputs (K, 1)
        inner_lr: Inner loop learning rate (alpha)
        inner_steps: Number of gradient steps
        create_graph: Whether to create computation graph for second-order gradients
        
    Returns:
        List of adapted parameters
    """
    # Clone current parameters for adaptation
    adapted_params = [p.clone() for p in model.parameters()]
    
    for step in range(inner_steps):
        # Forward pass with adapted parameters
        # We need to manually apply parameters (functional approach)
        y_pred = model(x_support)
        loss = F.mse_loss(y_pred, y_support)
        
        # Compute gradients
        grads = torch.autograd.grad(
            loss, 
            model.parameters(), 
            create_graph=create_graph  # Critical for MAML!
        )
        
        # Update adapted parameters
        adapted_params = [p - inner_lr * g for p, g in zip(adapted_params, grads)]
        
        # Update model parameters temporarily
        for param, adapted in zip(model.parameters(), adapted_params):
            param.data = adapted.data
    
    return adapted_params

### Test the inner loop on a single task

Let's see if one task adaptation step improves predictions.

In [ ]:
# Create a fresh model
test_model = SineModel(hidden_size=40).to(device)

# Create a task
task = SineWaveTask(amplitude=2.0, phase=0.5)
x_support, y_support = task.sample(10)
x_query, y_query = task.sample(100)

x_support, y_support = x_support.to(device), y_support.to(device)
x_query, y_query = x_query.to(device), y_query.to(device)

# Predictions before adaptation
with torch.no_grad():
    y_pred_before = test_model(x_query)
    loss_before = F.mse_loss(y_pred_before, y_query).item()

# Adapt the model
adapted_params = inner_loop_adapt(
    test_model, x_support, y_support, 
    inner_lr=0.01, inner_steps=5, create_graph=False
)

# Predictions after adaptation
with torch.no_grad():
    y_pred_after = test_model(x_query)
    loss_after = F.mse_loss(y_pred_after, y_query).item()

print(f"Query loss before adaptation: {loss_before:.4f}")
print(f"Query loss after adaptation: {loss_after:.4f}")
print(f"Improvement: {(loss_before - loss_after) / loss_before * 100:.1f}%")

### Visualize adaptation

Let's see how the model's predictions change after adaptation.

In [ ]:
# Reset model
test_model = SineModel(hidden_size=40).to(device)

# Create plotting data
x_plot = np.linspace(-5, 5, 200)
x_plot_tensor = torch.tensor(x_plot, dtype=torch.float32).unsqueeze(1).to(device)

# Before adaptation
with torch.no_grad():
    y_pred_before = test_model(x_plot_tensor).cpu().numpy()

# Adapt
adapted_params = inner_loop_adapt(
    test_model, x_support, y_support,
    inner_lr=0.01, inner_steps=5, create_graph=False
)

# After adaptation
with torch.no_grad():
    y_pred_after = test_model(x_plot_tensor).cpu().numpy()

# Plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(x_plot, task.true_function(x_plot), 'k-', linewidth=2, label='True function')
plt.plot(x_plot, y_pred_before, 'r--', linewidth=2, label='Before adaptation')
plt.scatter(x_support.cpu().numpy(), y_support.cpu().numpy(), 
           c='blue', marker='o', s=100, edgecolors='black', linewidth=1.5,
           label='Support set', zorder=3)
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Before Adaptation (Random Initialization)', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-6, 6)

plt.subplot(1, 2, 2)
plt.plot(x_plot, task.true_function(x_plot), 'k-', linewidth=2, label='True function')
plt.plot(x_plot, y_pred_after, 'g-', linewidth=2, label='After adaptation')
plt.scatter(x_support.cpu().numpy(), y_support.cpu().numpy(), 
           c='blue', marker='o', s=100, edgecolors='black', linewidth=1.5,
           label='Support set', zorder=3)
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('After 5 Gradient Steps', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-6, 6)

plt.tight_layout()
plt.show()

print("Even from random initialization, the model can partially fit the task with just a few gradient steps.")
print("MAML will learn initial parameters that make this adaptation much more effective!")

### Outer Loop: Meta-Optimization

The **outer loop** optimizes the meta-parameters $\theta$ to maximize post-adaptation performance:

$$\theta \leftarrow \theta - \beta \nabla_{\theta} \sum_{\mathcal{T}_i} \mathcal{L}_{\text{query}}^{\mathcal{T}_i}(\theta'_i)$$

This is the magic of MAML: we're optimizing for good performance **after adaptation**, not before.

The gradient $\nabla_{\theta} \mathcal{L}_{\text{query}}(\theta')$ requires computing gradients through the inner loop adaptation — this is a **second-order gradient**.

In [ ]:
def maml_outer_step(
    model: nn.Module,
    tasks: List[SineWaveTask],
    inner_lr: float,
    inner_steps: int,
    k_shot: int,
    n_query: int
) -> float:
    """
    Perform one MAML outer loop step on a batch of tasks.
    
    Args:
        model: The meta-model to update
        tasks: Batch of tasks
        inner_lr: Inner loop learning rate
        inner_steps: Number of inner loop steps
        k_shot: Number of support examples per task
        n_query: Number of query examples per task
        
    Returns:
        Average query loss across tasks
    """
    meta_loss = 0.0
    
    for task in tasks:
        # Sample support and query sets
        x_support, y_support = task.sample(k_shot)
        x_query, y_query = task.sample(n_query)
        
        x_support, y_support = x_support.to(device), y_support.to(device)
        x_query, y_query = x_query.to(device), y_query.to(device)
        
        # Inner loop: adapt to this task
        adapted_params = inner_loop_adapt(
            model, x_support, y_support, 
            inner_lr, inner_steps, create_graph=True
        )
        
        # Evaluate on query set with adapted parameters
        y_pred_query = model(x_query)
        task_loss = F.mse_loss(y_pred_query, y_query)
        
        meta_loss += task_loss
    
    # Average over tasks
    meta_loss = meta_loss / len(tasks)
    
    return meta_loss

### Full MAML Training Loop

Now we'll combine inner and outer loops into the complete MAML algorithm.

In [ ]:
def train_maml(
    model: nn.Module,
    n_iterations: int,
    tasks_per_iteration: int,
    k_shot: int,
    n_query: int,
    inner_lr: float,
    outer_lr: float,
    inner_steps: int,
    device: torch.device
) -> List[float]:
    """
    Train a model using MAML.
    
    Returns:
        List of meta-training losses
    """
    # Outer loop optimizer
    meta_optimizer = torch.optim.Adam(model.parameters(), lr=outer_lr)
    
    losses = []
    
    for iteration in range(n_iterations):
        # Sample a batch of tasks
        tasks = [SineWaveTask() for _ in range(tasks_per_iteration)]
        
        # Zero gradients
        meta_optimizer.zero_grad()
        
        # Compute meta-loss
        meta_loss = maml_outer_step(
            model, tasks, inner_lr, inner_steps, k_shot, n_query
        )
        
        # Meta-gradient descent
        meta_loss.backward()
        meta_optimizer.step()
        
        losses.append(meta_loss.item())
        
        if (iteration + 1) % 100 == 0:
            print(f"Iteration {iteration + 1}/{n_iterations}, Meta-Loss: {meta_loss.item():.4f}")
    
    return losses

## 5. Training MAML

### Train the meta-learner

We'll train MAML with 5-shot learning. Each meta-iteration samples multiple sine wave tasks.

In [ ]:
# Initialize model
maml_model = SineModel(hidden_size=40).to(device)

# MAML hyperparameters
n_iterations = 500
tasks_per_iteration = 4
k_shot = 5  # 5 examples per task for adaptation
n_query = 10  # 10 examples for evaluation
inner_lr = 0.01  # Learning rate for task adaptation
outer_lr = 0.001  # Learning rate for meta-update
inner_steps = 1  # Number of gradient steps for adaptation

print("Training MAML...")
print(f"Meta-training setup: {tasks_per_iteration} tasks per iteration, {k_shot}-shot, {inner_steps} inner steps\n")

losses = train_maml(
    maml_model,
    n_iterations=n_iterations,
    tasks_per_iteration=tasks_per_iteration,
    k_shot=k_shot,
    n_query=n_query,
    inner_lr=inner_lr,
    outer_lr=outer_lr,
    inner_steps=inner_steps,
    device=device
)

print("\nTraining complete!")

### Plot meta-training curve

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(losses, linewidth=2)
plt.xlabel('Meta-Iteration', fontsize=12)
plt.ylabel('Meta-Loss (Query Set MSE)', fontsize=12)
plt.title('MAML Training Curve', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final meta-loss: {losses[-1]:.4f}")
print("The meta-loss measures average performance after adaptation across many tasks.")

## 6. Evaluating MAML

### Compare MAML vs Random Initialization

The key question: Does MAML's learned initialization adapt faster than a random initialization?

Let's test on a brand new task that wasn't seen during meta-training.

In [ ]:
# Create a new test task
test_task = SineWaveTask(amplitude=3.5, phase=1.2)
x_support, y_support = test_task.sample(k_shot)
x_query, y_query = test_task.sample(100)

x_support, y_support = x_support.to(device), y_support.to(device)
x_query, y_query = x_query.to(device), y_query.to(device)

# Test MAML-initialized model
maml_model_copy = deepcopy(maml_model)
losses_maml = []

# Measure loss before and after each adaptation step
for step in range(10):
    with torch.no_grad():
        y_pred = maml_model_copy(x_query)
        loss = F.mse_loss(y_pred, y_query).item()
        losses_maml.append(loss)
    
    # One adaptation step
    inner_loop_adapt(maml_model_copy, x_support, y_support, 
                    inner_lr=0.01, inner_steps=1, create_graph=False)

# Test random initialization
random_model = SineModel(hidden_size=40).to(device)
losses_random = []

for step in range(10):
    with torch.no_grad():
        y_pred = random_model(x_query)
        loss = F.mse_loss(y_pred, y_query).item()
        losses_random.append(loss)
    
    inner_loop_adapt(random_model, x_support, y_support,
                    inner_lr=0.01, inner_steps=1, create_graph=False)

# Plot comparison
plt.figure(figsize=(12, 5))
plt.plot(losses_maml, 'g-o', linewidth=2, markersize=6, label='MAML Initialization')
plt.plot(losses_random, 'r-s', linewidth=2, markersize=6, label='Random Initialization')
plt.xlabel('Adaptation Step', fontsize=12)
plt.ylabel('Query Set MSE', fontsize=12)
plt.title('Adaptation Speed: MAML vs Random', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f"MAML - Initial loss: {losses_maml[0]:.4f}, After 10 steps: {losses_maml[-1]:.4f}")
print(f"Random - Initial loss: {losses_random[0]:.4f}, After 10 steps: {losses_random[-1]:.4f}")
print(f"\nMAML achieves {losses_random[-1] / losses_maml[-1]:.1f}x lower loss after adaptation!")

### Visualize adaptation trajectory

Let's see how predictions evolve during adaptation for both initializations.

In [ ]:
# Create plotting data
x_plot = np.linspace(-5, 5, 200)
x_plot_tensor = torch.tensor(x_plot, dtype=torch.float32).unsqueeze(1).to(device)

# Fresh copies
maml_model_viz = deepcopy(maml_model)
random_model_viz = SineModel(hidden_size=40).to(device)

# Adaptation steps to visualize
steps_to_plot = [0, 1, 3, 9]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for step in range(10):
    if step in steps_to_plot:
        idx = steps_to_plot.index(step)
        
        # MAML predictions
        with torch.no_grad():
            y_pred_maml = maml_model_viz(x_plot_tensor).cpu().numpy()
            loss_maml = F.mse_loss(maml_model_viz(x_query), y_query).item()
        
        axes[0, idx].plot(x_plot, test_task.true_function(x_plot), 'k-', linewidth=2, label='True')
        axes[0, idx].plot(x_plot, y_pred_maml, 'g-', linewidth=2, label='MAML')
        axes[0, idx].scatter(x_support.cpu().numpy(), y_support.cpu().numpy(),
                           c='blue', marker='o', s=80, edgecolors='black', linewidth=1.5, zorder=3)
        axes[0, idx].set_title(f'MAML - Step {step}\nLoss: {loss_maml:.3f}', fontweight='bold')
        axes[0, idx].set_ylim(-6, 6)
        axes[0, idx].grid(True, alpha=0.3)
        axes[0, idx].legend(fontsize=9)
        
        # Random predictions
        with torch.no_grad():
            y_pred_random = random_model_viz(x_plot_tensor).cpu().numpy()
            loss_random = F.mse_loss(random_model_viz(x_query), y_query).item()
        
        axes[1, idx].plot(x_plot, test_task.true_function(x_plot), 'k-', linewidth=2, label='True')
        axes[1, idx].plot(x_plot, y_pred_random, 'r-', linewidth=2, label='Random')
        axes[1, idx].scatter(x_support.cpu().numpy(), y_support.cpu().numpy(),
                           c='blue', marker='o', s=80, edgecolors='black', linewidth=1.5, zorder=3)
        axes[1, idx].set_title(f'Random - Step {step}\nLoss: {loss_random:.3f}', fontweight='bold')
        axes[1, idx].set_ylim(-6, 6)
        axes[1, idx].grid(True, alpha=0.3)
        axes[1, idx].legend(fontsize=9)
    
    # Perform one adaptation step
    inner_loop_adapt(maml_model_viz, x_support, y_support,
                    inner_lr=0.01, inner_steps=1, create_graph=False)
    inner_loop_adapt(random_model_viz, x_support, y_support,
                    inner_lr=0.01, inner_steps=1, create_graph=False)

plt.tight_layout()
plt.show()

print("MAML starts much closer to the true function and adapts more smoothly.")
print("Random initialization requires many more steps to converge to a good solution.")

## 7. Understanding Second-Order Gradients

### Why Second-Order Gradients?

MAML's power comes from **second-order gradients** — gradients of gradients.

**First-order gradient:** $\nabla_{\theta} \mathcal{L}(\theta)$ tells us how to change $\theta$ to reduce loss.

**Second-order gradient:** $\nabla_{\theta} \mathcal{L}(\theta - \alpha \nabla_{\theta} \mathcal{L}(\theta))$ tells us how to change $\theta$ to reduce loss **after** a gradient step.

This is expensive to compute but crucial for MAML. The outer loop gradient must account for how changing $\theta$ affects:
1. The inner loop gradient $\nabla_{\theta} \mathcal{L}_{\text{support}}$
2. The adapted parameters $\theta'$
3. The query loss $\mathcal{L}_{\text{query}}(\theta')$

### First-Order MAML (Faster Approximation)

Computing second-order gradients is expensive. **First-Order MAML** (FOMAML) approximates by ignoring the gradient through the inner loop:

$$\nabla_{\theta} \mathcal{L}_{\text{query}}(\theta') \approx \nabla_{\theta'} \mathcal{L}_{\text{query}}(\theta')$$

This is much faster but slightly less accurate. In practice, FOMAML often works nearly as well as full MAML.

To implement FOMAML, we simply set `create_graph=False` in the inner loop and use `.detach()` on adapted parameters.

## 8. Alternative Meta-Learning Approaches

### Metric-Based Meta-Learning

Instead of learning good initial parameters (optimization-based), we can learn good **similarity metrics** (metric-based).

The idea: Learn an embedding space where:
- Examples from the same class are close together
- Examples from different classes are far apart

Then classify by comparing query examples to support examples using the learned metric.

**Key approaches:**

1. **Matching Networks** (Vinyals et al., 2016)
   - Uses attention over support set to classify queries
   - No fine-tuning needed — just compare embeddings

2. **Prototypical Networks** (Snell et al., 2017)
   - Compute class **prototypes** (mean embedding per class)
   - Classify by nearest prototype
   - Simple and effective!

### Implementing Prototypical Networks

Let's implement a simple version for our sine wave regression task (adapted to regression instead of classification).

In [ ]:
class ProtoNet(nn.Module):
    """Prototypical Network for regression."""
    
    def __init__(self, hidden_size: int = 40, embedding_size: int = 16):
        super().__init__()
        # Embedding network
        self.encoder = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, embedding_size)
        )
        
        # Prediction head
        self.predictor = nn.Sequential(
            nn.Linear(embedding_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    
    def forward(self, x):
        """Direct forward pass."""
        z = self.encoder(x)
        return self.predictor(z)
    
    def embed(self, x):
        """Get embeddings."""
        return self.encoder(x)
    
    def predict_from_embedding(self, z):
        """Predict from embedding."""
        return self.predictor(z)

### Train Prototypical Network

For regression, we'll use prototype-based predictions: average the embeddings of support examples, then use that as context.

In [ ]:
def train_protonet(
    model: ProtoNet,
    n_iterations: int,
    tasks_per_iteration: int,
    k_shot: int,
    n_query: int,
    lr: float,
    device: torch.device
) -> List[float]:
    """Train Prototypical Network."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    for iteration in range(n_iterations):
        total_loss = 0.0
        
        for _ in range(tasks_per_iteration):
            task = SineWaveTask()
            x_support, y_support = task.sample(k_shot)
            x_query, y_query = task.sample(n_query)
            
            x_support = x_support.to(device)
            y_support = y_support.to(device)
            x_query = x_query.to(device)
            y_query = y_query.to(device)
            
            # Simple approach: just train to predict directly
            # (A full prototypical network would use distance-based classification)
            optimizer.zero_grad()
            
            # Predict on support set
            y_pred_support = model(x_support)
            loss_support = F.mse_loss(y_pred_support, y_support)
            
            # Predict on query set
            y_pred_query = model(x_query)
            loss_query = F.mse_loss(y_pred_query, y_query)
            
            # Total loss
            loss = loss_support + loss_query
            loss.backward()
            optimizer.step()
            
            total_loss += loss_query.item()
        
        avg_loss = total_loss / tasks_per_iteration
        losses.append(avg_loss)
        
        if (iteration + 1) % 100 == 0:
            print(f"Iteration {iteration + 1}/{n_iterations}, Loss: {avg_loss:.4f}")
    
    return losses

### Train and compare with MAML

Let's see how Prototypical Networks compare to MAML on the same task distribution.

In [ ]:
proto_model = ProtoNet(hidden_size=40, embedding_size=16).to(device)

print("Training Prototypical Network...")
proto_losses = train_protonet(
    proto_model,
    n_iterations=500,
    tasks_per_iteration=4,
    k_shot=5,
    n_query=10,
    lr=0.001,
    device=device
)

print("\nTraining complete!")

### Compare training curves

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(losses, linewidth=2, label='MAML', alpha=0.8)
plt.plot(proto_losses, linewidth=2, label='Prototypical Network', alpha=0.8)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Meta-Loss (Query Set MSE)', fontsize=12)
plt.title('MAML vs Prototypical Networks', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Both approaches learn to solve the meta-learning problem, but through different mechanisms:")
print("- MAML: learns good initialization for gradient-based adaptation")
print("- Prototypical: learns good embedding space for direct prediction")

## 9. Meta-Learning vs Transfer Learning

### Key Differences

**Transfer Learning:**
- Pre-train on large source dataset
- Fine-tune on target task (one-time)
- Works when source and target are similar
- No explicit "learning to learn"

**Meta-Learning:**
- Train across many tasks simultaneously
- Learn to adapt quickly (repeatedly)
- Works across diverse task distributions
- Explicitly optimizes adaptation ability

**When to use each:**

Use **transfer learning** when:
- You have one target task with moderate data
- Source and target are closely related (e.g., ImageNet → object detection)
- You want simplicity

Use **meta-learning** when:
- You need to solve many related but different tasks
- Each task has very little data (1-10 examples)
- You want fast adaptation at test time
- Tasks share structure but differ in specifics

### Visualizing the difference

Let's compare how transfer learning and meta-learning approach the same few-shot problem.

In [ ]:
# Transfer learning: train on many tasks, then fine-tune on new task
transfer_model = SineModel(hidden_size=40).to(device)
transfer_optimizer = torch.optim.Adam(transfer_model.parameters(), lr=0.001)

# Pre-train on many tasks (like pre-training on ImageNet)
print("Pre-training transfer learning model on many tasks...")
for iteration in range(200):
    total_loss = 0.0
    for _ in range(4):
        task = SineWaveTask()
        x, y = task.sample(20)  # More data per task
        x, y = x.to(device), y.to(device)
        
        transfer_optimizer.zero_grad()
        y_pred = transfer_model(x)
        loss = F.mse_loss(y_pred, y)
        loss.backward()
        transfer_optimizer.step()
        
        total_loss += loss.item()
    
    if (iteration + 1) % 100 == 0:
        print(f"Iteration {iteration + 1}: Loss = {total_loss / 4:.4f}")

print("Pre-training complete!")

### Test adaptation on a new task

In [ ]:
# New test task
test_task = SineWaveTask(amplitude=2.5, phase=0.8)
x_support, y_support = test_task.sample(5)
x_query, y_query = test_task.sample(50)
x_support, y_support = x_support.to(device), y_support.to(device)
x_query, y_query = x_query.to(device), y_query.to(device)

# MAML adaptation
maml_test = deepcopy(maml_model)
maml_losses = []
for step in range(10):
    with torch.no_grad():
        loss = F.mse_loss(maml_test(x_query), y_query).item()
        maml_losses.append(loss)
    inner_loop_adapt(maml_test, x_support, y_support, 0.01, 1, False)

# Transfer learning fine-tuning
transfer_test = deepcopy(transfer_model)
transfer_opt = torch.optim.SGD(transfer_test.parameters(), lr=0.01)
transfer_losses = []
for step in range(10):
    with torch.no_grad():
        loss = F.mse_loss(transfer_test(x_query), y_query).item()
        transfer_losses.append(loss)
    
    # Fine-tune
    transfer_opt.zero_grad()
    loss = F.mse_loss(transfer_test(x_support), y_support)
    loss.backward()
    transfer_opt.step()

# Plot comparison
plt.figure(figsize=(12, 5))
plt.plot(maml_losses, 'g-o', linewidth=2, markersize=6, label='MAML (meta-learned)')
plt.plot(transfer_losses, 'b-s', linewidth=2, markersize=6, label='Transfer Learning (pre-trained)')
plt.xlabel('Adaptation Step', fontsize=12)
plt.ylabel('Query Set MSE', fontsize=12)
plt.title('Meta-Learning vs Transfer Learning Adaptation', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f"MAML final loss: {maml_losses[-1]:.4f}")
print(f"Transfer learning final loss: {transfer_losses[-1]:.4f}")
print("\nMAML adapts faster because it was trained to adapt, not just to solve individual tasks.")

## 10. Practical Considerations and Extensions

### Hyperparameters in MAML

MAML has several important hyperparameters:

1. **Inner learning rate** ($\alpha$): Too high → unstable adaptation; too low → slow adaptation
2. **Outer learning rate** ($\beta$): Standard meta-learning rate
3. **Inner steps**: More steps → better adaptation but more expensive computation
4. **Tasks per meta-batch**: More tasks → more stable meta-gradient but slower

**Typical ranges:**
- Inner LR: 0.001 - 0.1
- Outer LR: 0.0001 - 0.001
- Inner steps: 1-10
- Tasks per batch: 4-32

### Real-World Applications

**Where meta-learning shines:**

1. **Few-shot image classification**: Recognize new object categories from 1-5 examples
   - Medical imaging: rare diseases with limited scans
   - Wildlife monitoring: identify species from few photos

2. **Personalization**: Adapt models to individual users
   - Recommendation systems with new users
   - Speech recognition for new speakers

3. **Robotics**: Learn new motor skills quickly
   - Adapt to new environments or objects
   - Transfer across different robot morphologies

4. **Drug discovery**: Predict properties of new molecules
   - Limited experimental data per compound
   - Need to generalize across molecular structures

### Extensions and Variants

**Beyond basic MAML:**

1. **MAML++** (Antoniou et al., 2019)
   - Multi-step loss optimization
   - Per-parameter learning rates
   - Improved training stability

2. **Reptile** (Nichol et al., 2018)
   - First-order approximation of MAML
   - Even simpler: just average adapted parameters
   - Often works as well as MAML

3. **Meta-SGD** (Li et al., 2017)
   - Learn inner learning rates as well as initialization
   - More flexible adaptation

4. **ANIL** (Almost No Inner Loop, Raghu et al., 2020)
   - Only adapt the final layer in inner loop
   - Much faster with similar performance

5. **Task-conditional architectures**
   - FiLM layers: modulate features based on task embedding
   - Hypernetworks: generate task-specific weights

### Challenges and Limitations

**Current limitations:**

1. **Computational cost**: Second-order gradients are expensive
   - Solution: Use FOMAML or Reptile

2. **Task distribution matters**: MAML learns from task distribution
   - Doesn't generalize to very different task distributions
   - Need sufficient task diversity during meta-training

3. **Instability**: Inner loop can be unstable with poor hyperparameters
   - Solution: Careful tuning, gradient clipping, MAML++

4. **Memory requirements**: Must store computation graph for inner loop
   - Limits batch size and inner steps

5. **Not always better**: For tasks with moderate data, standard fine-tuning may suffice
   - Meta-learning overhead is only worth it for truly few-shot scenarios

## 11. Key Takeaways

### Core Concepts

**Meta-learning is "learning to learn":**
- Train on a distribution of tasks to learn how to adapt quickly
- Evaluate adaptation ability on query sets
- Two-level optimization: inner loop (adaptation) + outer loop (meta-learning)

**N-way K-shot setup:**
- N classes, K examples per class in support set
- Support set for adaptation, query set for evaluation
- Each episode is a different task

**MAML learns initial parameters that are easy to fine-tune:**
- Inner loop: adapt to specific task via gradient descent
- Outer loop: update initialization to improve post-adaptation performance
- Requires second-order gradients (gradients through gradients)

**Metric-based alternatives learn good similarity functions:**
- Prototypical Networks: classify via nearest prototype
- Matching Networks: attention-weighted nearest neighbors
- No gradient-based adaptation needed at test time

**Meta-learning ≠ transfer learning:**
- Meta-learning: optimize for fast adaptation across task distribution
- Transfer learning: pre-train once, fine-tune once
- Meta-learning better for diverse tasks with very limited data

**Key design choices:**
- Inner/outer learning rates affect adaptation speed and stability
- Number of inner steps trades off adaptation quality vs computation
- Task diversity during meta-training determines generalization
- First-order approximations (FOMAML, Reptile) often work well

**Practical impact:**
- Enables learning from 1-10 examples per class
- Useful for personalization, rare categories, and rapid adaptation
- Active research area with many extensions and applications